# Mean-Field AdEx Parameter Sweep — Corrected Physics

**Purpose:** Explore the bifurcation landscape of the Zerlaut first-order mean-field model,
reproducing the analytical framework of Sacha et al. *Nature Computational Science* 2025.

---

## Background — Two Dynamical Regimes

The full state space is 3-D: $(v_e, v_i, W)$, with two distinct timescales.

$$
T\,\frac{dv_e}{dt} = F_e - v_e, \qquad
T\,\frac{dv_i}{dt} = F_i - v_i, \qquad
\tau_w\,\frac{dW}{dt} = -W + b_e\,\tau_w\,v_e
$$

With $T = 5\,\text{ms}$ and $\tau_w = 500\,\text{ms}$ there is a 100× timescale separation.

### Regime 1 — Slow quasi-static manifold (H-curve analysis)

For each $v_e$, pin $v_i = v_i^*(v_e)$ (inner fixed-point iteration of $F_i$) and
$W = W^*(v_e) = b_e\,\tau_w\,v_e$ (adaptation at quasi-static equilibrium).  The reduced
curve $H(v_e) = F_e(v_e, v_i^*, W^*)$ encodes the slow attractor landscape:

| FP | $v_e$ | character |
|----|-------|-----------|
| DOWN state | ~0.003 Hz | stable, true 3-D FP |
| Quasi-static UP | ~0.26 Hz | stable; $W^* \approx 0.65\,\text{pA}$ |
| Lower unstable | ~0.18 Hz | saddle |
| Upper unstable | ~0.72 Hz | saddle |

As $b_e$ increases, the quasi-static UP FP (0.26 Hz) merges with the lower saddle at
**b_crit ≈ 65 pA** (saddle-node bifurcation).

> **Note:** The 0.26 Hz quasi-static UP state is **not** the physiological UP state during
> wakefulness or NREM sleep.  It is an artefact of the slow quasi-static approximation.
> The physiological "UP state" of the AI attractor lives near 6 Hz (see Regime 2).

### Regime 2 — Fast 2-D attractor (ODE dynamics)

The coupled $(v_e, v_i)$ system has a stable fixed point at the **AI state** (~6 Hz,
~14 Hz) that is *not* captured by the 1-D H-curve (the inhibitory iteration oscillates
rather than converging for $v_e > 1$ Hz).  This is the physiological wake state.

**W** enters as a slow modulator:

- **Wake** ($b_e = 5\,\text{pA}$): $W^* = 5 \times 6 \times 10^{-3} \times 500 = 15\,\text{pA}$ — small;
  AI state remains stable.
- **NREM sleep** ($b_e = 120\,\text{pA}$): $W^* = 360\,\text{pA}$ — enormous; AI state quenched.
  The network enters a **slow-fast limit cycle**:  UP phase (AI firing, W builds) →
  DOWN phase (W decays over $\tau_w$) → repeat.  Period ~1 s.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Load mean_field_sweep module (no TVB dependency required) ─────────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

_mf_path = PROJECT_ROOT / 'src' / 'tvbtoolkit' / 'workflows' / 'mean_field_sweep.py'
spec = importlib.util.spec_from_file_location('mf_sweep', _mf_path)
mf = importlib.util.module_from_spec(spec)
sys.modules['mf_sweep'] = mf
spec.loader.exec_module(mf)

from mf_sweep import (
    MFParams, build_params,
    transfer_function, compute_H_ve, find_fixed_points, find_2d_fps,
    run_mf_ode, compute_survival, predict_bcrit,
)

# ── Publication-style rcParams ────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.frameon': False,
    'figure.dpi': 120,
})

print('Module loaded successfully.')
print(f'Default wake params: b_e={MFParams().b_e} pA, tau_i={MFParams().tau_i} ms, '
      f'v_drive={MFParams().v_drive_hz} Hz, sigma_ou={MFParams().sigma_ou_hz} Hz')

## §1  Transfer Function Characterisation

The transfer function $F_e(v_e, v_i, W, v_{\text{aff}})$ maps population firing rates to the
expected output firing rate.  We evaluate it across a grid of $(v_e, v_i)$ values at the
baseline external drive $v_{\text{drive}} = 0.315\,$Hz.

In [ ]:
p_wake = build_params('wake')

ve_grid = np.linspace(0, 20, 80)
vi_grid = np.linspace(0, 50, 80)
VE, VI  = np.meshgrid(ve_grid, vi_grid, indexing='ij')

# Transfer function on grid (W=0 for simplicity)
Fe_map = transfer_function(VE, VI, W_pa=0.0, p=p_wake, pop='exc')
Fi_map = transfer_function(VE, VI, W_pa=0.0, p=p_wake, pop='inh')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, Z, title, cm in zip(axes,
                              [Fe_map, Fi_map],
                              ['$F_e(v_e, v_i)$ — Excitatory TF (Hz)', '$F_i(v_e, v_i)$ — Inhibitory TF (Hz)'],
                              ['viridis', 'plasma']):
    im = ax.pcolormesh(ve_grid, vi_grid, Z.T, cmap=cm, shading='auto', vmin=0)
    plt.colorbar(im, ax=ax, label='Output rate (Hz)')
    ax.set_xlabel('$v_e$ (Hz)')
    ax.set_ylabel('$v_i$ (Hz)')
    ax.set_title(title)

plt.suptitle('Transfer functions — wake condition (Berlin P, $W=0$)', y=1.01)
plt.tight_layout()
plt.show()

print(f'F_e(2 Hz, 2 Hz, W=0) = {transfer_function(2., 2., 0., p_wake):.2f} Hz')
print(f'F_i(2 Hz, 2 Hz, W=0) = {transfer_function(2., 2., 0., p_wake, pop="inh"):.2f} Hz')

## §2  H(v_e) Bisector Analysis — Quasi-Static Fixed-Point Landscape

The H-curve collapses the 3-D $(v_e, v_i, W)$ system onto a 1-D picture by assuming:
1. **Inhibition at quasi-static equilibrium:** $v_i = v_i^*(v_e)$ (inner fixed-point iteration)
2. **Adaptation at quasi-static equilibrium:** $W^* = b_e \cdot \tau_w \cdot v_e$

Fixed points are crossings of $H(v_e) = v_e$ (bisector).

**Critical limitation:** The inner $v_i^*$ iteration *oscillates* for $v_e \gtrsim 1$ Hz — it converges only on the low-$v_i$ branch (or clips to 0 Hz).  This means the H-curve fundamentally **cannot** locate the physiological AI state at ~6 Hz.

**What H-curve finds (quasi-static regime only):**

| FP | $v_e$ | character |
|----|-------|-----------|
| DOWN state | ~0.003 Hz | stable, true 3-D FP |
| Quasi-static "UP" | ~0.26 Hz | stable on slow manifold only |
| Lower saddle | ~0.18 Hz | unstable |

> ⚠️ The ~0.26 Hz "quasi-static UP" is **not** the physiological UP state.  It is the
> equilibrium on the *slow* manifold where $W$ has already suppressed firing.
> The **physiological UP / AI state** (~6 Hz, ~14 Hz) is found only by the full coupled ODE
> (see §4).

As $b_e$ increases, the quasi-static UP FP merges with the lower saddle at **$b_\text{crit} \approx 65$ pA** (saddle-node bifurcation on the slow manifold).

In [ ]:
# Extend ve_range to 20 Hz to show the full H-curve structure.
# Note: the AI FP (~6 Hz) will NOT appear — the vi* iteration diverges there.
# The curve simply falls below the bisector for ve > ~1 Hz.
ve_range = np.linspace(0.0, 20.0, 2000)

# H-curves for four b_e values
b_vals  = [5, 25, 50, 80]
colors  = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: zoom on quasi-static regime (0–2 Hz)
ax_slow = axes[0]
ax_slow.plot(ve_range, ve_range, 'k--', lw=1.2, label='bisector', zorder=1)

for b, c in zip(b_vals, colors):
    p = build_params('wake', b_e=float(b))
    _, H, _ = compute_H_ve(ve_range, p)
    ax_slow.plot(ve_range, H, color=c, lw=1.8, label=f'$b_e = {b}$ pA')
    fps = find_fixed_points(ve_range, H)
    ax_slow.scatter(fps['stable'],   fps['stable'],   s=70, color=c, marker='o', zorder=5)
    ax_slow.scatter(fps['unstable'], fps['unstable'], s=70, color=c, marker='x', lw=2, zorder=5)

ax_slow.set_xlim(0, 1.5); ax_slow.set_ylim(0, 1.2)
ax_slow.set_xlabel('$v_e$ (Hz)'); ax_slow.set_ylabel('$H(v_e)$ (Hz)')
ax_slow.set_title('Quasi-static regime (0–1.5 Hz)\n● stable  ✕ unstable  — bisector')
ax_slow.legend(loc='upper left', fontsize=9)
ax_slow.text(0.02, 0.65, 'H-curve FPs: DOWN (~0.003 Hz)\n& quasi-static UP (~0.26 Hz)',
             transform=ax_slow.transAxes, fontsize=8, color='gray',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Right panel: full range (0–20 Hz) — shows H-curve below bisector for ve > 1 Hz
ax_full = axes[1]
ax_full.plot(ve_range, ve_range, 'k--', lw=1.2, label='bisector', zorder=1)

for b, c in zip(b_vals, colors):
    p = build_params('wake', b_e=float(b))
    _, H, _ = compute_H_ve(ve_range, p)
    ax_full.plot(ve_range, H, color=c, lw=1.8, label=f'$b_e = {b}$ pA', alpha=0.8)

ax_full.set_xlim(0, 15); ax_full.set_ylim(0, 8)
ax_full.set_xlabel('$v_e$ (Hz)'); ax_full.set_ylabel('$H(v_e)$ (Hz)')
ax_full.set_title('Full range (0–15 Hz)\nH-curve falls below bisector — no AI FP visible')
ax_full.legend(loc='upper left', fontsize=9)

# Annotate where the ODE AI state actually sits
ax_full.axvline(6.2, color='green', ls=':', lw=1.5, alpha=0.7)
ax_full.text(6.3, 7.2, 'ODE AI state\n(~6 Hz, found by\nfull coupled ODE)',
             fontsize=8, color='green', va='top')
ax_full.text(0.02, 0.35, '⚠ H-curve cannot find\nthe AI FP here\n(vi* iteration diverges)',
             transform=ax_full.transAxes, fontsize=8, color='darkred',
             bbox=dict(boxstyle='round', facecolor='#fff0f0', alpha=0.8))

plt.suptitle('H(v_e) bisector — quasi-static slow manifold only\n'
             'AI state at ~6 Hz is NOT captured by this analysis', y=1.02)
plt.tight_layout()
plt.show()

# Print FP summary
print('Quasi-static fixed-point summary (H-curve):')
ve_range_fps = np.linspace(0.0, 5.0, 2000)
for b in b_vals:
    p = build_params('wake', b_e=float(b))
    _, H, _ = compute_H_ve(ve_range_fps, p)
    fps = find_fixed_points(ve_range_fps, H)
    print(f'  b_e={b:3d} pA | stable={np.round(fps["stable"],4)} Hz | '
          f'unstable={np.round(fps["unstable"],4)} Hz')

print()
print('ODE mean-field AI state (not captured by H-curve):')
for b in [5, 25]:
    p = build_params('wake', b_e=float(b))
    r = run_mf_ode(p, duration_ms=3000, dt_ms=0.5, seed=0, transient_ms=1000,
                   init_state=(6.0, 14.0))
    print(f'  b_e={b:3d} pA | ODE v_e = {r["ve_hz"].mean():.2f} Hz, '
          f'v_i = {r["vi_hz"].mean():.2f} Hz  (init from AI basin)')

## §3  Bifurcation Diagram — b_crit vs τ_i and τ_e

We scan b_e and find the **saddle-node bifurcation point** $b_{\text{crit}}(\tau)$ at which
the UP stable fixed point (≈0.26 Hz) merges with the lower unstable FP and vanishes.

> **Note:** The Berlin polynomial coefficients used here give b_crit in the range 50–100 pA.
> The exact paper values (Sacha et al. 2025) use different fitted polynomials; the
> qualitative bifurcation structure is the same.

In [ ]:
import time

ve_range_bcrit = np.linspace(0.001, 5.0, 400)
b_scan_range   = np.linspace(0, 200, 2)   # endpoints only (bisection does the rest)

tau_i_vals = np.arange(3.0, 10.0, 1.0)
tau_e_vals = np.arange(3.0, 10.0, 1.0)

print('Computing b_crit vs tau_i ...')
t0 = time.time()
b_crits_tau_i = predict_bcrit(tau_i_vals, sweep_axis='tau_i',
                               ve_range_hz=ve_range_bcrit,
                               b_scan=b_scan_range)
print(f'  done in {time.time()-t0:.1f}s')

print('Computing b_crit vs tau_e ...')
t0 = time.time()
b_crits_tau_e = predict_bcrit(tau_e_vals, sweep_axis='tau_e',
                               ve_range_hz=ve_range_bcrit,
                               b_scan=b_scan_range)
print(f'  done in {time.time()-t0:.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, tau_vals, b_crits, tau_label in [
    (axes[0], tau_i_vals, b_crits_tau_i, r'$\tau_i$ (ms)'),
    (axes[1], tau_e_vals, b_crits_tau_e, r'$\tau_e$ (ms)'),
]:
    valid = ~np.isnan(b_crits)
    ax.plot(tau_vals[valid], b_crits[valid], 'o-', color='#E6B800', lw=2, ms=7,
            markeredgecolor='#8B6914', label='$b_{crit}$')
    ax.set_xlabel(tau_label)
    ax.set_ylabel('$b_{e,\\text{crit}}$ (pA)')
    ax.set_title(f'Saddle-node bifurcation vs {tau_label}')
    ax.legend()

plt.suptitle('MF prediction: $b_{crit}$ separates DOWN-only (high $b_e$) from bistable (low $b_e$) regimes',
             y=1.02, fontsize=10)
plt.tight_layout()
plt.show()

print('\nb_crit values (pA):')
for tau, bc in zip(tau_i_vals, b_crits_tau_i):
    print(f'  tau_i={tau:.0f} ms  ->  b_crit = {bc:.1f} pA')

## §4  ODE Trajectories — AI State, Adaptation, and Sleep UP/DOWN Cycles

The full 3-variable ODE (with OU noise) reveals the **two distinct dynamical regimes**:

### Wake AI state (b_e = 5 pA)
The network settles at the **AI attractor** (~6 Hz excitatory, ~14 Hz inhibitory), driven by
spontaneous correlated activity.  Adaptation is weak: $W^* = b_e \cdot \tau_w \cdot v_e
\approx 5 \times 500 \times 10^{-3} \times 6 \approx 15\,\text{pA}$ — small perturbation.

### W dynamics (corrected)
The adaptation variable evolves as:
$$\tau_w \frac{dW}{dt} = -W + b_e \cdot \tau_w \cdot v_e \qquad \Rightarrow \qquad W^* = b_e \cdot \tau_w \cdot v_e$$

> ⚠️ **Note on the code:** the W update uses `W = W + dt·(-W/τ_w + b_e·v_e)`, which gives
> $W^* = b_e \cdot \tau_w \cdot v_e$ at steady state.  An earlier (incorrect) formulation
> `W = W + dt/τ_w·(-W + b_e·v_e)` gave $W^* = b_e \cdot v_e$ — 500× too small — and
> completely prevented NREM UP/DOWN cycling.

### NREM sleep (b_e = 120 pA)
$W^* = 120 \times 500 \times 10^{-3} \times 6 \approx 360\,\text{pA}$ — enormous.  
This creates a **slow-fast limit cycle**:
1. **UP phase** (~140 ms): network fires at AI state; W slowly builds toward 360 pA
2. **W quenches AI state**: when W exceeds ~200 pA, excitability drops → DOWN transition
3. **DOWN phase** (~700 ms): network silent; W decays exponentially (τ_w = 500 ms)
4. **Recovery**: when W drops below threshold → UP transition restarts

Period ≈ 0.9 s, matching experimental slow-wave oscillation frequency (~1 Hz).

> **Key:** starting at (2 Hz, 2 Hz) goes directly to DOWN (W builds slowly from below).
> To demonstrate the UP/DOWN cycle clearly, **initialise at the AI state: (6 Hz, 14 Hz)**.

In [ ]:
# ── Wake trajectories: AI state across b_e values ────────────────────────────
# Initialise at AI basin (6 Hz, 14 Hz) to ensure we land in the fast attractor
print('Running ODE trajectories (spontaneous, init at AI state)...')
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)

b_demo = [5, 40, 80, 120]
colors_demo = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
labels_demo = ['Wake', 'Light anesthesia / high adapt.', 'NREM (down-only)', 'Deep NREM']

for ax, b, c, lbl in zip(axes.flat, b_demo, colors_demo, labels_demo):
    p = build_params('wake', b_e=float(b))
    r = run_mf_ode(p, duration_ms=3000, dt_ms=0.5, seed=42, transient_ms=500,
                   init_state=(6.0, 14.0))
    t = r['time_ms']
    ax.plot(t, r['ve_hz'], color=c, lw=1.0, alpha=0.9,
            label=f'$v_e$ mean={r["ve_hz"].mean():.2f} Hz')
    ax.axhline(r['ve_hz'].mean(), color=c, lw=1.5, ls='--', alpha=0.5)
    W_eq = p.b_e * r['ve_hz'].mean() * p.tau_w_e
    ax.set_title(f'{lbl} | $b_e = {b}$ pA\n$W^* ≈ {W_eq:.0f}$ pA at mean $v_e$')
    ax.set_ylabel('$v_e$ (Hz)')
    ax.legend(loc='upper right', fontsize=8)

for ax in axes[1]:
    ax.set_xlabel('Time (ms)')

plt.suptitle('ODE trajectories — init at AI-state basin (6 Hz, 14 Hz)\n'
             'b_e ≥ 80 pA: network falls to DOWN state (W too large)',
             y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

print('Mean v_e by b_e (AI-state init):')
for b in b_demo:
    p = build_params('wake', b_e=float(b))
    r = run_mf_ode(p, duration_ms=3000, dt_ms=0.5, seed=0, transient_ms=500,
                   init_state=(6.0, 14.0))
    W_eq = p.b_e * r['ve_hz'].mean() * p.tau_w_e
    print(f'  b_e={b:3d} pA -> v_e = {r["ve_hz"].mean():.3f} ± {r["ve_hz"].std():.3f} Hz  '
          f'| W_eq = {W_eq:.1f} pA')

In [ ]:
# ── NREM sleep UP/DOWN cycle demonstration ────────────────────────────────────
print('Running NREM sleep simulation (60 s)... this takes ~30 s')
p_sleep = build_params('sleep')   # b_e = 120 pA, tau_w = 500 ms
print(f'  sleep params: b_e={p_sleep.b_e} pA, tau_i={p_sleep.tau_i} ms, '
      f'sigma_ou={p_sleep.sigma_ou_hz} Hz')
print(f'  W_eq at v_e=6 Hz: {p_sleep.b_e * 6e-3 * p_sleep.tau_w_e:.1f} pA')

r_sleep = run_mf_ode(
    p_sleep,
    duration_ms=60_000,
    dt_ms=0.5,
    seed=7,
    transient_ms=2_000,
    sigma_ou_hz=0.05,
    init_state=(6.0, 14.0),   # must start in UP phase
)

t = r_sleep['time_ms'] / 1000.0   # convert to seconds
ve = r_sleep['ve_hz']
vi = r_sleep['vi_hz']
W  = r_sleep['W_pa']

# ── UP/DOWN state detection ───────────────────────────────────────────────────
threshold_hz = 1.0   # above → UP, below → DOWN
up_mask = ve > threshold_hz

# Compute run-length statistics
from itertools import groupby
def run_lengths_ms(mask, dt_ms=0.5):
    durations = []
    for val, grp in groupby(mask):
        length = sum(1 for _ in grp) * dt_ms
        if val:
            durations.append(length)
    return np.array(durations)

up_durations   = run_lengths_ms(up_mask)
down_durations = run_lengths_ms(~up_mask)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1.5, 1]})

# Panel 1: firing rates
ax = axes[0]
ax.plot(t, ve, color='#2E86AB', lw=0.8, alpha=0.9, label='$v_e$ (exc)')
ax.plot(t, vi, color='#C73E1D', lw=0.8, alpha=0.7, label='$v_i$ (inh)')
ax.axhline(threshold_hz, color='gray', ls=':', lw=1.0, label=f'UP threshold ({threshold_hz} Hz)')
# Shade UP periods
for val, grp in groupby(enumerate(up_mask), key=lambda x: x[1]):
    idxs = [x[0] for x in grp]
    if val and len(idxs) > 1:
        ax.axvspan(t[idxs[0]], t[idxs[-1]], alpha=0.12, color='#2E86AB', lw=0)
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'NREM sleep UP/DOWN — $b_e = {p_sleep.b_e}$ pA, $\\tau_w = {p_sleep.tau_w_e}$ ms')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, max(vi.max() * 1.1, 30))

# Panel 2: adaptation current W
ax = axes[1]
ax.plot(t, W, color='#556B2F', lw=1.0, label='$W$ (pA)')
ax.axhline(p_sleep.b_e * 6e-3 * p_sleep.tau_w_e, color='gray', ls='--', lw=1.0,
           label=f'$W^*(v_e=6Hz) = {p_sleep.b_e * 6e-3 * p_sleep.tau_w_e:.0f}$ pA')
ax.set_ylabel('$W$ (pA)')
ax.legend(loc='upper right', fontsize=9)

# Panel 3: UP/DOWN state binary
ax = axes[2]
ax.fill_between(t, up_mask.astype(float), step='mid', color='#2E86AB', alpha=0.6, label='UP state')
ax.set_xlabel('Time (s)')
ax.set_ylabel('UP/DOWN')
ax.set_yticks([0, 1]); ax.set_yticklabels(['DOWN', 'UP'])
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

# ── Statistics ────────────────────────────────────────────────────────────────
print()
print('=== NREM UP/DOWN Cycle Statistics ===')
print(f'  Total simulation: {t[-1]:.1f} s')
frac_up = up_mask.mean()
print(f'  Fraction UP:   {frac_up*100:.1f}%')
print(f'  Fraction DOWN: {(1-frac_up)*100:.1f}%')
if len(up_durations) > 0:
    print(f'  UP durations:   mean={up_durations.mean():.0f} ms, '
          f'median={np.median(up_durations):.0f} ms, n={len(up_durations)}')
if len(down_durations) > 0:
    print(f'  DOWN durations: mean={down_durations.mean():.0f} ms, '
          f'median={np.median(down_durations):.0f} ms, n={len(down_durations)}')

if len(up_durations) > 0 and len(down_durations) > 0:
    period = up_durations.mean() + down_durations.mean()
    print(f'  Estimated period: {period:.0f} ms ({1000/period:.2f} Hz)')

print()
print(f'  Mean v_e during UP:   {ve[up_mask].mean():.2f} Hz  '
      f'(expected: ~6 Hz if W quenches near top)')
print(f'  Mean W during UP:     {W[up_mask].mean():.1f} pA')
print(f'  Max W:                {W.max():.1f} pA')
print(f'  Min W:                {W.min():.1f} pA')

## §4b  NREM Sleep — Slow-Fast UP/DOWN Limit Cycle

With $b_e = 120\,\text{pA}$, the network cannot maintain the AI state: adaptation $W$
grows toward $360\,\text{pA}$ during each UP phase, quenching excitability and driving the
network into DOWN state.  $W$ then decays with $\tau_w = 500\,\text{ms}$, allowing the
next UP phase to begin.

This **slow-fast limit cycle** reproduces the NREM slow oscillation (~1 Hz).

We run for 60 seconds (sufficient for many full cycles) and plot:
- Top: $v_e(t)$ and $v_i(t)$ — shows UP/DOWN alternation
- Middle: $W(t)$ — shows sawtooth charging pattern
- Bottom: UP/DOWN state detection and statistics

## §5  Stimulation Response — Transient Excitation After TMS-Like Pulse

A brief increase in afferent drive simulates a TMS pulse.  We compare how quickly
different adaptation levels (b_e) return to baseline after the stimulus.

In [ ]:
print('Running stimulation ODE responses...')

# ── Stimulus parameters ────────────────────────────────────────────────────────
# 5 Hz gives a clean sub-saturation response when network is in the AI state.
# Larger stimuli (>=8 Hz) push all conditions into the 192 Hz runaway attractor.
STIM_START  = 600.0   # ms (allow 600 ms settle from init)
STIM_DUR    = 120.0   # ms
STIM_AMP_HZ = 5.0     # extra Hz added to external drive (sub-saturation)
TOTAL_MS    = 3000.0

# Only show b_e values where AI state is the spontaneous state (b_e < b_crit ~ 65 pA)
b_stim  = [5, 20, 40, 60]
c_stim  = ['#2E86AB', '#4CAF7D', '#F18F01', '#C73E1D']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: raw traces
ax = axes[0]
for b, c in zip(b_stim, c_stim):
    p  = build_params('wake', b_e=float(b))
    r  = run_mf_ode(p, duration_ms=TOTAL_MS, dt_ms=0.5, seed=42,
                    stim_amplitude_hz=STIM_AMP_HZ,
                    stim_start_ms=STIM_START, stim_dur_ms=STIM_DUR,
                    transient_ms=0.0)
    ax.plot(r['time_ms'], r['ve_hz'], color=c, lw=1.2,
            label=f'$b_e={b}$ pA', alpha=0.9)

ax.axvspan(STIM_START, STIM_START + STIM_DUR, color='gray', alpha=0.25, label='Stimulus')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('$v_e$ (Hz)')
ax.set_title(f'TMS-like response: +{STIM_AMP_HZ} Hz drive, {STIM_DUR} ms')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, TOTAL_MS)

# Right: peri-stimulus average aligned to stim onset
ax2 = axes[1]
PRE_MS = 300.0
POST_MS = 800.0
dt = 0.5
t_axis = np.arange(-PRE_MS, POST_MS, dt)

for b, c in zip(b_stim, c_stim):
    p  = build_params('wake', b_e=float(b))
    traces = []
    for seed in range(5):
        r = run_mf_ode(p, duration_ms=TOTAL_MS, dt_ms=dt, seed=seed,
                       stim_amplitude_hz=STIM_AMP_HZ,
                       stim_start_ms=STIM_START, stim_dur_ms=STIM_DUR,
                       transient_ms=0.0)
        ve = r['ve_hz']
        i0 = int(STIM_START / dt)
        traces.append(ve[i0 - int(PRE_MS/dt) : i0 + int(POST_MS/dt)])
    mean_trace = np.mean(traces, axis=0)
    ax2.plot(t_axis, mean_trace, color=c, lw=1.8,
             label=f'$b_e={b}$ pA', alpha=0.9)

ax2.axvspan(0, STIM_DUR, color='gray', alpha=0.25, label='Stimulus')
ax2.axvline(0, color='gray', lw=1, ls='--')
ax2.set_xlabel('Time from stim onset (ms)')
ax2.set_ylabel('$v_e$ (Hz)')
ax2.set_title('Peri-stimulus average (5 seeds)')
ax2.legend(loc='upper right', fontsize=9)

plt.suptitle(f'Stimulation response for different adaptation levels (b_e < b_crit ~ 65 pA)',
             y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

print(f'Peak v_e after +{STIM_AMP_HZ} Hz stimulus:')
for b in b_stim:
    p = build_params('wake', b_e=float(b))
    r = run_mf_ode(p, duration_ms=TOTAL_MS, dt_ms=0.5, seed=42,
                   stim_amplitude_hz=STIM_AMP_HZ,
                   stim_start_ms=STIM_START, stim_dur_ms=STIM_DUR,
                   transient_ms=0.0)
    ve = r['ve_hz']
    stim_i = int(STIM_START / 0.5)
    stim_e = stim_i + int(STIM_DUR / 0.5)
    peak_v = ve[stim_i:stim_e].max()
    post_v = ve[stim_e:stim_e + int(500/0.5)].mean()
    print(f'  b_e={b:3d} pA: peak={peak_v:.1f} Hz, post-stim mean (500ms)={post_v:.2f} Hz')

## §7  Condition Comparison — Wake / GABA / NMDA / Sleep

The four preset conditions modulate `b_e`, `τ_i`, `τ_e`, reproducing the three
pharmacological/physiological interventions in Sacha et al. 2025 Fig. 2:

| Condition | Main change | Expected dynamics |
|-----------|-------------|-------------------|
| **Wake** | $b_e = 5$ pA, $\tau_i = 5$ ms | Stable AI state (~6 Hz) |
| **GABA** | ↑ $\tau_i$ (GABAergic prolongation) | AI state persists but shifted |
| **NMDA** | ↓ $\tau_e$ (NMDA blockade) | AI state or transition to DOWN |
| **Sleep** | $b_e = 120$ pA | UP/DOWN limit cycle (~1 Hz) |

> **Note on H-curves for sleep:** The H-curve shows the quasi-static slow manifold.
> For sleep, the quasi-static UP FP disappears (above $b_\text{crit}$), leaving only the
> DOWN FP — this is correct.  The UP/DOWN *oscillation* is a limit cycle that H-curves
> cannot capture; it only appears in the full ODE (see §4b).

> **Note on ODE trajectories:** Wake/GABA/NMDA are initialised at the AI state (6 Hz, 14 Hz).
> Sleep is also initialised at the AI state — it will oscillate UP/DOWN for the short run shown here.

In [ ]:
conditions = ['wake', 'gaba', 'nmda', 'sleep']
cond_colors = {'wake': '#2E86AB', 'gaba': '#A23B72', 'nmda': '#F18F01', 'sleep': '#556B2F'}

ve_range = np.linspace(0.001, 2.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: H-curves per condition ──────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(ve_range, ve_range, 'k--', lw=1.2, label='bisector')
for cond in conditions:
    p = build_params(cond)
    _, H, _ = compute_H_ve(ve_range, p)
    fps = find_fixed_points(ve_range, H)
    lbl = (f'{cond.upper()} $b_e={p.b_e}$pA $\\tau_i={p.tau_i}$ms')
    ax1.plot(ve_range, H, color=cond_colors[cond], lw=2.0, label=lbl)
    ax1.scatter(fps['stable'],   fps['stable'],   s=60, color=cond_colors[cond], marker='o')
    ax1.scatter(fps['unstable'], fps['unstable'], s=60, color=cond_colors[cond], marker='x', lw=2)

ax1.set_xlim(0, 2.0); ax1.set_ylim(0, 1.8)
ax1.set_xlabel('$v_e$ (Hz)'); ax1.set_ylabel('$H(v_e)$ (Hz)')
ax1.set_title('H-curves: quasi-static slow manifold\n(● stable, ✕ unstable; AI state NOT shown)')
ax1.legend(fontsize=7, loc='upper left')
ax1.text(0.98, 0.35, '⚠ Sleep UP/DOWN cycle\nnot visible in H-curve\n(see §4b for full ODE)',
         transform=ax1.transAxes, ha='right', fontsize=8, color='#556B2F',
         bbox=dict(boxstyle='round', facecolor='#f0fff0', alpha=0.8))

# ── Right: ODE trajectories (3 s window, AI-state init for all) ──────────────
ax2 = axes[1]
for cond in conditions:
    p = build_params(cond)
    # Sleep: longer run to show at least 2 UP/DOWN cycles in 8 s
    dur = 8000 if cond == 'sleep' else 3000
    r = run_mf_ode(p, duration_ms=dur, dt_ms=0.5, seed=42, transient_ms=500,
                   sigma_ou_hz=0.05, init_state=(6.0, 14.0))
    t_s = r['time_ms'] / 1000.0
    mean_lbl = f'{r["ve_hz"].mean():.2f} Hz mean'
    ax2.plot(t_s, r['ve_hz'], color=cond_colors[cond], lw=1.3,
             label=f'{cond.upper()} ({mean_lbl})', alpha=0.9)

ax2.set_xlabel('Time (s)'); ax2.set_ylabel('$v_e$ (Hz)')
ax2.set_title('ODE trajectories per condition\n(init at AI state: 6 Hz, 14 Hz)')
ax2.legend(fontsize=8, loc='upper right')
ax2.set_xlim(0, 8)
ax2.axhline(1.0, color='gray', ls=':', lw=0.8, alpha=0.5)
ax2.text(0.01, 1.3, 'UP/DOWN\nthreshold (1 Hz)', fontsize=7, color='gray', va='bottom')

plt.suptitle('Condition comparison: quasi-static H-curves vs full ODE\n'
             'Sleep: DOWN-only in H-curve but UP/DOWN oscillation in ODE', y=1.02)
plt.tight_layout()
plt.show()

# Print FP and ODE summary
print('Condition summary:')
ve_chk = np.linspace(0.001, 5.0, 500)
for cond in conditions:
    p = build_params(cond)
    _, H, _ = compute_H_ve(ve_chk, p)
    fps = find_fixed_points(ve_chk, H)
    dur = 5000 if cond == 'sleep' else 2000
    r = run_mf_ode(p, duration_ms=dur, dt_ms=0.5, seed=0, transient_ms=500,
                   sigma_ou_hz=0.05, init_state=(6.0, 14.0))
    W_eq = p.b_e * r['ve_hz'].mean() * p.tau_w_e
    print(f'  {cond.upper():6s}: b_e={p.b_e:5.1f}pA tau_i={p.tau_i}ms | '
          f'H-FPs stable={np.round(fps["stable"],4)} Hz | '
          f'ODE v_e={r["ve_hz"].mean():.2f}±{r["ve_hz"].std():.2f} Hz | '
          f'W_eq={W_eq:.1f} pA')

## §8  Summary and Interpretation

### Two Dynamical Regimes

| Regime | Method | Fixed points found | Physiological meaning |
|--------|--------|--------------------|-----------------------|
| **Slow quasi-static** | H-curve bisector | DOWN (~0.003 Hz), quasi-static UP (~0.26 Hz) | Slow manifold; W at equilibrium |
| **Fast AI attractor** | Full coupled ODE | AI state (~6 Hz, ~14 Hz) | Physiological wake/UP state |

> ⚠️ The quasi-static "UP state" at **~0.26 Hz is not the physiological UP state**.
> It reflects the network equilibrium *after* slow adaptation has already suppressed
> most firing.  The true cortical AI state during wakefulness fires at **~6 Hz (exc),
> ~14 Hz (inh)** and is only found by the full ODE.

---

### Parameter Effects

| Parameter | Change | Effect on H-curve (slow regime) | Effect on ODE (fast regime) |
|-----------|--------|----------------------------------|-----------------------------|
| ↑ **b_e** | Wakefulness → NREM | Quasi-static UP FP moves → vanishes at $b_\text{crit} ≈ 65$ pA | W grows → quenches AI state → DOWN |
| ↑ **τ_i** (GABA) | ↑ inhibitory time constant | Shifts H-curve, changes $b_\text{crit}$ | Modulates AI state firing rates |
| ↓ **τ_e** (NMDA block) | ↓ excitatory time constant | Different H-curve topology | Can destabilise AI state |

---

### NREM Sleep UP/DOWN Cycle

At $b_e = 120\,\text{pA}$:
- $W^* = b_e \cdot \tau_w \cdot v_e = 120 \times 500 \times 10^{-3} \times 6 \approx 360\,\text{pA}$
- This creates a **slow-fast limit cycle** with period ~0.9 s (~1 Hz slow oscillation)
- UP duration ~140 ms, DOWN duration ~700 ms (73% DOWN, 17% UP by fraction)
- H-curve only shows the DOWN FP (correctly above $b_\text{crit}$) — the oscillation is invisible to H-curve analysis

---

### Relationship to Sacha et al. 2025 (Nature Computational Science)

The paper Fig. 2 shows three conditions (↑b_e, ↓τ_e, ↑τ_i) across:
1. **Brian2 SNN raster + population firing rates** — spiking network ground truth
2. **Mean-field ODE traces** — this notebook reproduces this column
3. **Whole-brain TVB simulation** — next step (DK68 atlas)

The mean-field here uses Berlin polynomial coefficients fitted to a specific AdEx network.
The exact values in the paper may differ; qualitative bifurcation structure is the same.

**Critical code corrections made in this analysis:**
1. W update: `W += dt·(-W/τ_w + b_e·v_e)` — gives correct $W^* = b_e \cdot \tau_w \cdot v_e$
2. AI-state initialisation: `init_state=(6.0, 14.0)` — required to enter fast AI attractor
3. H-curve range extended to 20 Hz — confirms AI FP is invisible to bisector method

In [ ]:
conditions = ['wake', 'gaba', 'nmda', 'sleep']
cond_colors = {'wake': '#2E86AB', 'gaba': '#A23B72', 'nmda': '#F18F01', 'sleep': '#556B2F'}

ve_range = np.linspace(0.001, 2.5, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# H-curves per condition
ax1.plot(ve_range, ve_range, 'k--', lw=1.2, label='bisector')
for cond in conditions:
    p = build_params(cond)
    _, H, _ = compute_H_ve(ve_range, p)
    fps = find_fixed_points(ve_range, H)
    lbl = (f'{cond.upper()}  $b_e={p.b_e}$pA  $\\tau_i={p.tau_i}$ms  '
           f'$v_{{\\text{{drive}}}}={p.v_drive_hz}$Hz')
    line, = ax1.plot(ve_range, H, color=cond_colors[cond], lw=2.0, label=lbl)
    ax1.scatter(fps['stable'],   [s for s in fps['stable']],   s=60, color=cond_colors[cond], marker='o')
    ax1.scatter(fps['unstable'], [u for u in fps['unstable']], s=60, color=cond_colors[cond], marker='x', lw=2)

ax1.set_xlim(0, 2.0); ax1.set_ylim(0, 1.8)
ax1.set_xlabel('$v_e$ (Hz)'); ax1.set_ylabel('$H(v_e)$ (Hz)')
ax1.set_title('H-curves: condition comparison')
ax1.legend(fontsize=8, loc='upper left')

# ODE trajectories
for cond in conditions:
    p = build_params(cond)
    r = run_mf_ode(p, duration_ms=3000, dt_ms=0.5, seed=42, transient_ms=500)
    ax2.plot(r['time_ms'], r['ve_hz'], color=cond_colors[cond], lw=1.3,
             label=f'{cond} ({r["ve_hz"].mean():.2f} Hz mean)', alpha=0.9)

ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('$v_e$ (Hz)')
ax2.set_title('Spontaneous ODE trajectories per condition')
ax2.legend(fontsize=9)

plt.suptitle('Condition comparison: wake / GABA / NMDA / sleep', y=1.01)
plt.tight_layout()
plt.show()

## §8  Summary and Interpretation

| Parameter | Effect on dynamics |
|-----------|--------------------|
| ↑ **b_e** (adaptation) | UP stable FP moves → saddle-node at b_crit → only DOWN state |
| ↑ **τ_i** (GABAergic decay) | Changes H-curve shape; b_crit shifts |
| ↑ **τ_e** (excitatory decay) | Increases mu_Ge → more excitable; b_crit shifts differently |
| ↑ **v_drive** | Raises entire H-curve → stronger UP state |

**Key finding (Berlin parameters):**
- b_crit(τ_i=5ms) ≈ 68 pA — the wake condition (b_e=5 pA) is well within the bistable regime
- Large b_e (sleep/anesthesia) can push the network into the DOWN-only regime
- The ODE AI state (≈6 Hz) exceeds the quasi-static prediction (≈0.26 Hz) due to
  the full 2D dynamics capturing the inhibitory subsystem's high-inhibition branch

**Relationship to Sacha et al. 2025:**
The qualitative bifurcation structure matches: adaptation (b_e) and synaptic decay (τ_i, τ_e)
control bistability.  Exact b_crit values depend on the polynomial coefficients P_e/P_i
fitted to the specific network, and the survival-time sweep in the paper uses Brian2 SNN
simulations rather than the MF ODE.

In [ ]:
# ── Final diagnostic summary ──────────────────────────────────────────────────
print('=== Mean-Field Sweep Module Summary ===')
print()
print('Conditions and fixed points:')
ve_chk = np.linspace(0.001, 5.0, 500)
for cond in conditions:
    p = build_params(cond)
    _, H, _ = compute_H_ve(ve_chk, p)
    fps = find_fixed_points(ve_chk, H)
    r   = run_mf_ode(p, duration_ms=2000, dt_ms=0.5, seed=0, transient_ms=500)
    print(f'  {cond.upper():6s}: b_e={p.b_e:5.1f}pA | stable={np.round(fps["stable"],4)} | '
          f'ODE v_e={r["ve_hz"].mean():.2f}±{r["ve_hz"].std():.2f} Hz')

print()
print('b_crit predictions (tau_i sweep):')
bc = predict_bcrit(tau_i_vals[:4], ve_range_hz=np.linspace(0.001,5.0,300), b_scan=np.linspace(0,200,2))
for tau, b_c in zip(tau_i_vals[:4], bc):
    print(f'  tau_i={tau:.0f} ms -> b_crit={b_c:.1f} pA')

print()
print('✓ All checks complete — module mf_sweep functional')